# EXPERIMENT 4A – Clustering Analysis using K-Means on Iris Dataset

**Aim:** To implement K-Means clustering on the Iris dataset and determine the optimum number of clusters using the Elbow Method.

**Dataset:** Iris.csv – [Kaggle Iris Dataset](https://www.kaggle.com/datasets/uciml/iris)

**Algorithm:** K-Means clustering.

**Note:** K-Means is an unsupervised learning algorithm. The species column is not used while forming clusters; it can optionally be used later to compare the clusters with the known species labels.

## 1. Import required libraries
Pandas is used to read the CSV file, Matplotlib to plot the elbow curve, and Scikit-learn to scale data and apply K-Means.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import preprocessing and clustering tools
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Load the Iris dataset

Download `Iris.csv` from Kaggle and place it in the same folder as this notebook. If it is stored elsewhere, update the file path in `pd.read_csv()`.

In [ ]:
# Load the Iris CSV file
df = pd.read_csv("Iris.csv")

# Display the first five rows
display(df.head())

# Display dataset size and column information
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())

## 3. Prepare the data
We remove identifier columns such as `Id` and the target label such as `Species`. Only the four numerical flower measurements are used for clustering.

In [ ]:
# Select only numerical measurement columns
# This excludes the Id column and the Species label automatically.
X = df.select_dtypes(include=np.number).copy()

# Remove an identifier column if it is present
if "Id" in X.columns:
    X = X.drop(columns=["Id"])

# Fill any missing numeric values with the column median
X = X.fillna(X.median())

print("Features used for clustering:", X.columns.tolist())
display(X.head())

## 4. Feature scaling
Scaling is important because K-Means uses distances. StandardScaler converts each feature to a comparable scale.

In [ ]:
# Standardize the four numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)

## 5. Find the optimum number of clusters using the Elbow Method
For each value of K, K-Means calculates the within-cluster sum of squares (inertia). We plot inertia against K. The point where the decrease starts becoming slower is called the elbow.

In [ ]:
# Store the inertia for each tested number of clusters
inertia = []
k_values = range(1, 11)

for k in k_values:
    # Create and fit a K-Means model for this K
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)

    # Inertia is the sum of squared distances to cluster centers
    inertia.append(kmeans.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertia, marker="o")
plt.title("Elbow Method for Iris Dataset")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (Inertia)")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

## 6. Select K and train the final K-Means model

The Iris dataset commonly shows an elbow around **K = 3**. Inspect the curve from the previous cell and use the elbow value. For this practical, we use K = 3.

In [ ]:
# Choose the number of clusters based on the elbow plot
optimal_k = 3

# Create and train the final K-Means model
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add the cluster number to the original dataframe
df["Cluster"] = clusters

print("K-Means clustering completed.")
print("\nNumber of records in each cluster:")
print(df["Cluster"].value_counts().sort_index())

## 7. Visualize the clusters
For a simple two-dimensional plot, we display the first two scaled features. The clustering itself uses all numerical measurement features.

In [ ]:
# Plot clusters using the first two scaled features
plt.figure(figsize=(7, 5))

for cluster_id in range(optimal_k):
    # Select rows belonging to the current cluster
    points = X_scaled[clusters == cluster_id]

    plt.scatter(
        points[:, 0],
        points[:, 1],
        label=f"Cluster {cluster_id}"
    )

# Plot cluster centers for the same two features
centers = kmeans.cluster_centers_
plt.scatter(
    centers[:, 0],
    centers[:, 1],
    marker="X",
    s=200,
    label="Centroids"
)

plt.title("K-Means Clusters on Iris Dataset")
plt.xlabel(X.columns[0] + " (scaled)")
plt.ylabel(X.columns[1] + " (scaled)")
plt.legend()
plt.grid(True)
plt.show()

## 8. Evaluate clustering using Silhouette Score (optional)
The silhouette score ranges from -1 to 1. A higher value generally indicates better separation between clusters. It is an additional evaluation measure; the required K selection method in this experiment is the Elbow Method.

In [ ]:
# Calculate silhouette score for the chosen clustering
score = silhouette_score(X_scaled, clusters)
print("Silhouette Score:", round(score, 4))

## 9. Conclusion

K-Means clustering was successfully implemented on the Iris dataset. The Elbow Method was used to study the within-cluster sum of squares for different values of K. Based on the elbow curve, three clusters were selected and the data points were assigned to clusters.

**Result:** The elbow curve and cluster visualization are displayed when the notebook is executed.